# Week 10 - LangChain Basics
College Assistant Chatbot using LangChain and Ollama


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
import sys
from pathlib import Path
import pandas as pd
PROJECT_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))
from langchain_chatbot import ask_college_bot, summarize_text, answer_from_context


## 1. Test the chatbot with 20 queries


In [ ]:
queries = pd.read_csv(PROJECT_DIR / 'dataset' / 'user queries.csv')
results = []
for _, row in queries.iterrows():
    answer = ask_college_bot(row['user_query'], 'general')
    results.append({'user_query': row['user_query'], 'response': answer})
    print('USER:', row['user_query'])
    print('BOT:', answer, '\\n')
responses = pd.DataFrame(results)
responses.to_csv(PROJECT_DIR / 'dataset' / 'chatbot_responses.csv', index=False)
responses.head()


## 2. Test the five prompt templates


In [ ]:
from langchain_chatbot import chatbot_chains
sample_question = 'How can I get a bonafide certificate?'
for name, chain in chatbot_chains.items():
    print('TEMPLATE:', name)
    print(chain.invoke({'question': sample_question}), '\\n')


## 2A. Compare the original and improved prompt


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
model = ChatOllama(model='llama3.2', temperature=0.2)
original_prompt = ChatPromptTemplate.from_template('Answer this college question: {question}')
improved_prompt = ChatPromptTemplate.from_template('You are a polite college assistant. Give a clear answer in short steps. Do not guess official dates, fees or rules. If details are unknown, direct the student to the relevant college office. Question: {question}')
original_chain = original_prompt | model | StrOutputParser()
improved_chain = improved_prompt | model | StrOutputParser()
comparison_question = 'How can I apply for admission?'
print('ORIGINAL PROMPT OUTPUT:')
print(original_chain.invoke({'question': comparison_question}))
print('\\nIMPROVED PROMPT OUTPUT:')
print(improved_chain.invoke({'question': comparison_question}))


## 3. Summarization chain


In [ ]:
sample_text = '''The college library provides textbooks, reference books, newspapers and digital learning resources. Students should use their college ID while borrowing books. Contact the librarian for borrowing limits, due dates and fine details.'''
print(summarize_text(sample_text))


## 4. Question-answering chain


In [ ]:
context = '''Students can borrow books from the college library using their college ID card. For due dates and borrowing limits, students should contact the librarian.'''
question = 'What should students use to borrow books?'
print(answer_from_context(context, question))


## 5. View saved CSV files


In [ ]:
print('Chatbot responses:')
display(pd.read_csv(PROJECT_DIR / 'dataset' / 'chatbot_responses.csv').tail())
print('Chain results:')
display(pd.read_csv(PROJECT_DIR / 'dataset' / 'chain results.csv').tail())
